# 09 — V3: train lại từ PhoWhisper gốc trên dữ liệu V3

- ViMedCSS chia lại theo video (`data/ViMedCSS_vsplit`, 0 video chung giữa TRAIN/DEV/TEST). Split chính thức có 81,7% clip TRAIN
  thuộc video cũng nằm trong DEV/TEST/HARD, nên số ViMedCSS của V1/V2 lạc quan. **So V3 với B0V**, không so ViMedCSS với V1/V2.
- Loại khỏi TRAIN: clip im lặng và nhãn dày hơn 16 token/giây (`data/v3_train_exclude.json`).
- LR 3e-4 (V1 dùng 1e-3 bị nổ loss), 3 epoch (~2800 bước), eval DEV-subset batch 8.
- `DUTY = 1.0`: GPU chạy liên tục, không nghỉ (V1/V2 dùng 0,7). Giới hạn VRAM 4,5 GB giữ nguyên.

Chạy cell thiết lập, cell dữ liệu, cell kiểm tra, rồi bấm ▶ đúng cell tác vụ. Mỗi lần chỉ một job GPU. Interrupt dừng tiến trình con;
chạy lại cell train với cùng `RUN_ID` **và cùng `DUTY`** sẽ tiếp tục từ checkpoint gần nhất (đổi DUTY thì run bị từ chối vì cấu hình khác).
Giữ VS Code mở trong lúc train. Config: `ASR/experiments/V3_phowhisper_small_lora_vsplit/config.yaml`.


In [1]:
from pathlib import Path
import sys, json
here = Path.cwd().resolve()
PROJECT = next((p for p in [here, *here.parents] if (p/'asr/config.py').is_file()), None)
if PROJECT is None:
    PROJECT = next((p/'ASR' for p in [here, *here.parents] if (p/'ASR/asr/config.py').is_file()), None)
assert PROJECT is not None, 'Mở notebook từ trong workspace dự án'
PYTHON = PROJECT.parent/'.venv-asr/Scripts/python.exe'
assert PYTHON.is_file(), 'Chạy ASR/local/setup.ps1 trước'
sys.path.insert(0, str(PROJECT/'local'))
from notebook_process import run_cli as _run_cli
def run_cli(arguments):
    return _run_cli(PROJECT, PYTHON, arguments)
RUNNER = PROJECT/'scripts/local_run.py'

EXP = 'V3_phowhisper_small_lora_vsplit'
BASELINE = 'B0V_phowhisper_small_zeroshot_vsplit'
RUN_ID = 'local_v3'
DUTY = '1.0'
RUN = PROJECT/'local/runs'/EXP/RUN_ID
FINAL = RUN/'final'
DATA = PROJECT/'data'

def gpu_job_running():
    """PID của job GPU local đang chạy (giữ local/gpu.lock), hoặc None."""
    import psutil
    lock = PROJECT/'local/gpu.lock'
    if not lock.exists():
        return None
    pid = int(lock.read_text())
    return pid if psutil.pid_exists(pid) else None

def latest_checkpoint(run):
    ckpts = sorted((run/'checkpoints').glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
    return ckpts[-1] if ckpts else None

print('Run:', RUN)
print('Checkpoint gần nhất:', latest_checkpoint(RUN) or 'chưa có (train sẽ bắt đầu từ bước 0)')
print('Job GPU đang chạy:', gpu_job_running() or 'không')


Run: D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\runs\V3_phowhisper_small_lora_vsplit\local_v3
Checkpoint gần nhất: chưa có (train sẽ bắt đầu từ bước 0)
Job GPU đang chạy: không


## Dữ liệu V3

Chỉ dùng CPU. Nếu đã có `data/ViMedCSS_vsplit` thì cell chỉ in báo cáo; dữ liệu gốc `data/ViMedCSS` không bị sửa.


In [2]:
if not (DATA/'ViMedCSS_vsplit/split_report.json').is_file():
    run_cli([PROJECT/'scripts/build_v3_data.py'])
report = json.loads((DATA/'ViMedCSS_vsplit/split_report.json').read_text(encoding='utf-8'))
print('ViMedCSS_vsplit:', json.dumps(report['splits'], ensure_ascii=False))
print('Video chung giữa các split:', report['video_overlap'])
print('Bỏ: trùng id', len(report['dropped_duplicate_ids']), '| rỗng/im lặng', len(report['dropped_silent_ids']),
      '| loại khỏi TRAIN', report['train_exclude_counts'])


ViMedCSS_vsplit: {"train": {"clips": 12688, "hours": 24.82, "videos": 2076}, "dev": {"clips": 1523, "hours": 2.988, "videos": 259}, "test": {"clips": 1539, "hours": 2.911, "videos": 234}, "hard": {"clips": 71, "hours": 0.13, "videos": 40}}
Video chung giữa các split: {'train&dev': 0, 'train&test': 0, 'dev&test': 0}
Bỏ: trùng id 4 | rỗng/im lặng 64 | loại khỏi TRAIN {'VIVOS': 1, 'ViMedCSS_vsplit': 13}


In [3]:
run_cli([RUNNER, 'check', EXP, '--run-id', RUN_ID])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" check V3_phowhisper_small_lora_vsplit --run-id local_v3
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261024_154832_990266.log
[INIT] mode=check experiment=V3_phowhisper_small_lora_vsplit; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 0.7,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.989728768,
  "system_ram_ava

## 1. Train

Smoke 2 bước trước, rồi train đầy đủ: ~2800 bước, ~17–18 giây/bước ở DUTY 1,0, cộng ~28 lần eval → khoảng 16 giờ. Cắm sạc, tắt sleep.


In [4]:
run_cli([RUNNER, 'train', EXP, '--run-id', 'smoke_v3', '--smoke', '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" train V3_phowhisper_small_lora_vsplit --run-id smoke_v3 --smoke --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261024_154839_159470.log
[INIT] mode=train experiment=V3_phowhisper_small_lora_vsplit; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.989

In [5]:
pid = gpu_job_running()
if pid:
    print(f'Đang có job GPU khác (PID {pid}); không khởi chạy thêm. Đợi job đó xong hoặc dừng nó trước.')
else:
    print('Train V3: bắt đầu mới hoặc tiếp tục từ', latest_checkpoint(RUN) or 'bước 0', flush=True)
    run_cli([RUNNER, 'train', EXP, '--run-id', RUN_ID, '--duty-cycle', DUTY])
    print('Adapter tốt nhất:', FINAL if (FINAL/'adapter_model.safetensors').is_file() else 'chưa có')


Train V3: bắt đầu mới hoặc tiếp tục từ bước 0
[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" train V3_phowhisper_small_lora_vsplit --run-id local_v3 --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261024_154859_493188.log
[INIT] mode=train experiment=V3_phowhisper_small_lora_vsplit; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": 

## Theo dõi

Chỉ đọc file. DEV-subset của V3 khác V1/V2 (ViMedCSS mới), nên `eval_wer` ở đây không so trực tiếp với 0,1339 của V1.


In [6]:
ckpt = latest_checkpoint(RUN)
if ckpt:
    state = json.loads((ckpt/'trainer_state.json').read_text(encoding='utf-8'))
    print('Checkpoint gần nhất:', ckpt.name, '| bước', state['global_step'], '/', state['max_steps'])
    for e in state['log_history']:
        if 'eval_wer' in e:
            print(f"  bước {e['step']:5d}  eval_wer={e['eval_wer']:.4f}  eval_loss={e['eval_loss']:.4f}")
logs = sorted((PROJECT/'local/logs').glob('*.log'))
mine = [p for p in logs if f'mode=train experiment={EXP}' in p.read_text(encoding='utf-8', errors='replace')[:300]]
if mine:
    lines = mine[-1].read_text(encoding='utf-8', errors='replace').replace('\r', '\n').splitlines()
    progress = [l for l in lines if l.startswith('[TRAIN] ') and 'microbatch' not in l]
    print('\nLog:', mine[-1])
    print(*(progress[-3:] or lines[-5:]), sep='\n')


Checkpoint gần nhất: checkpoint-2802 | bước 2802 / 2802
  bước   100  eval_wer=0.1534  eval_loss=0.3757
  bước   200  eval_wer=0.1540  eval_loss=0.3375
  bước   300  eval_wer=0.1470  eval_loss=0.3291
  bước   400  eval_wer=0.1458  eval_loss=0.3130
  bước   500  eval_wer=0.1374  eval_loss=0.3002
  bước   600  eval_wer=0.1390  eval_loss=0.2983
  bước   700  eval_wer=0.1288  eval_loss=0.2916
  bước   800  eval_wer=0.1439  eval_loss=0.2860
  bước   900  eval_wer=0.1307  eval_loss=0.2845
  bước  1000  eval_wer=0.1308  eval_loss=0.2860
  bước  1100  eval_wer=0.1298  eval_loss=0.2886
  bước  1200  eval_wer=0.1261  eval_loss=0.2868
  bước  1300  eval_wer=0.1296  eval_loss=0.2837
  bước  1400  eval_wer=0.1201  eval_loss=0.2753
  bước  1500  eval_wer=0.1298  eval_loss=0.2755
  bước  1600  eval_wer=0.1231  eval_loss=0.2758
  bước  1700  eval_wer=0.1228  eval_loss=0.2751
  bước  1800  eval_wer=0.1199  eval_loss=0.2731
  bước  1900  eval_wer=0.1242  eval_loss=0.2939
  bước  2000  eval_wer=0.1275  e

## 2. Đánh giá trên DEV mới

V3 và B0V đều giải mã với guard (bỏ clip im lặng, giới hạn token theo độ dài audio). TEST/HARD vẫn khóa.
B0V chạy được ngay cả khi V3 chưa xong (khi GPU rảnh).


In [7]:
if not (FINAL/'adapter_model.safetensors').is_file():
    print('Chưa có', FINAL, '- train V3 xong trước.')
else:
    run_cli([RUNNER, 'eval', EXP, '--run-id', 'smoke_local_v3', '--max-per-source', '2', '--checkpoint', FINAL,
             '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V3_phowhisper_small_lora_vsplit --run-id smoke_local_v3 --max-per-source 2 --checkpoint "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\runs\V3_phowhisper_small_lora_vsplit\local_v3\final" --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261025_035238_976836.log
[INIT] mode=eval experiment=V3_phowhisper_small_lora_vsplit; importing PyTorch/ASR libraries...
[RUNNING] 10s elapsed; waiting for the next model/data update.
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
   

In [8]:
if not (FINAL/'adapter_model.safetensors').is_file():
    print('Chưa có', FINAL, '- train V3 xong trước.')
else:
    run_cli([RUNNER, 'eval', EXP, '--run-id', 'dev_local_v3', '--checkpoint', FINAL, '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V3_phowhisper_small_lora_vsplit --run-id dev_local_v3 --checkpoint "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\runs\V3_phowhisper_small_lora_vsplit\local_v3\final" --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261025_035319_717129.log
[INIT] mode=eval experiment=V3_phowhisper_small_lora_vsplit; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,

In [9]:
run_cli([RUNNER, 'eval', BASELINE, '--run-id', 'dev_guard', '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval B0V_phowhisper_small_zeroshot_vsplit --run-id dev_guard --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261025_044954_483593.log
[INIT] mode=eval experiment=B0V_phowhisper_small_zeroshot_vsplit; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.98

In [10]:
import pandas as pd
rows = {}
for name, path in [('B0V', PROJECT/'local/runs'/BASELINE/'dev_guard/eval/dev/metrics.json'),
                   ('V3', RUN.parent/'dev_local_v3/eval/dev/metrics.json')]:
    if path.is_file():
        m = json.loads(path.read_text(encoding='utf-8'))
        rows[name] = {'overall': 100*m['overall']['wer'],
                      **{s: 100*v['wer'] for s, v in m['by_source'].items()},
                      'term recall': 100*m['overall'].get('cs_term_recall', float('nan'))}
display(pd.DataFrame(rows).round(2).rename_axis('WER % (DEV V3, guarded decoding)'))


,B0V,V3
"WER % (DEV V3, guarded decoding)",,
overall,24.83,18.26
FLEURS,10.89,8.63
VIVOS,0.42,2.05
ViMedCSS_vsplit,29.05,16.59
VietMed,25.87,21.63
term recall,9.05,60.11


## 3. (Tùy chọn) V1/V2/B0/B1 với guard trên DEV cũ

Không ảnh hưởng V3. Mỗi model ~1 giờ; chạy khi GPU rảnh. Cho biết guard (bỏ clip im lặng, giới hạn token) thay đổi số cũ bao nhiêu.


In [11]:
for exp in ['V1G_phowhisper_small_lora_guard', 'V2G_phowhisper_small_lora_cont_guard',
            'B0G_phowhisper_small_zeroshot_guard', 'B1G_whisper_small_zeroshot_guard']:
    run_cli([RUNNER, 'eval', exp, '--run-id', 'dev_guard', '--duty-cycle', DUTY])


[START] "D:\1.FPT_FALL26\DPL - Deep Learning\.venv-asr\Scripts\python.exe" -u "D:\1.FPT_FALL26\DPL - Deep Learning\ASR\scripts\local_run.py" eval V1G_phowhisper_small_lora_guard --run-id dev_guard --duty-cycle 1.0
[LOG] D:\1.FPT_FALL26\DPL - Deep Learning\ASR\local\logs\20261025_054846_776863.log
[INIT] mode=eval experiment=V1G_phowhisper_small_lora_guard; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  "missing_data": [],
  "profile": {
    "total_gpu_budget_gb": 4.5,
    "gpu_duty_cycle": 1.0,
    "host_ram_budget_gb": 0,
    "save_steps": 100,
    "checkpoint_interval_seconds": 600,
    "save_total_limit": 2,
    "batch_size": 1,
    "grad_accum": 32,
    "num_workers": 0
  },
  "local_v1_final": false,
  "system_ram_total_gb": 16.989728768,
 